# Topic 3: GC Tuning & Serialization

> **Phase 4 → Week 6 → Topic 3**

---

## Why GC Hurts Spark

Spark runs on the JVM. JVM garbage collection (GC) periodically **stops all threads** to reclaim unused memory — a "stop-the-world" pause.

```
Normal execution:  ████████████████████████████████
With GC pauses:    ████████ [GC 500ms] ███████ [GC 1.2s] ████████

Impact on Spark:
• Long GC → task takes longer → stage takes longer (the slowest task)
• Very long GC → executor heartbeat timeout → driver marks executor dead
• GC during shuffle → other executors waiting → idle cluster
• Repeated full GC → throughput drops by 50-90%
```

## GC Generations

```
JVM Heap:
┌──────────────────────────────────────────────────────┐
│  Young Generation          │  Old Generation          │
│  ┌────────┬──────┬───────┐ │                          │
│  │ Eden   │ S0   │ S1    │ │  Long-lived objects       │
│  │ (new   │(from)│(to)   │ │  (cached RDDs, shuffle    │
│  │ objects)│     │       │ │   results, long tasks)    │
│  └────────┴──────┴───────┘ │                          │
│  Minor GC: fast (ms)       │  Major/Full GC: SLOW (s) │
└──────────────────────────────────────────────────────┘

Problem: Spark tasks create many short-lived objects (rows, intermediate
results) that quickly fill Eden → frequent minor GC. Large caches fill
Old Gen → expensive full GC.
```

---

## Interview Cheat Sheet

**Q: How do you reduce GC overhead in Spark?**
> Five approaches: (1) Use G1GC garbage collector (handles large heaps better than ParallelGC); (2) Enable off-heap memory for Tungsten — moves sort buffers and hash tables out of GC-managed heap; (3) Use Kryo serialization instead of Java — smaller objects, less heap pressure; (4) Reduce caching — unpersist DataFrames when no longer needed; (5) Increase `spark.memory.fraction` to give Spark more heap space, reducing the frequency of GC-triggering full heap.

**Q: What is the difference between Java serialization and Kryo?**
> Java serialization is the default — it's flexible (works with any Java object) but slow and produces large byte arrays. Kryo is a faster, more compact binary serializer — typically 2-5x smaller output and much faster. Configure with `spark.serializer=org.apache.spark.serializer.KryoSerializer`. You need to register custom classes with `spark.kryo.classesToRegister` for maximum efficiency.

**Q: What is G1GC and why is it preferred for Spark?**
> G1 (Garbage-First) GC divides the heap into equal-sized regions and can compact them incrementally. Unlike ParallelGC (default for large heaps), G1 provides more predictable pause times and handles heaps > 4GB much better. Enable with `-XX:+UseG1GC`. For Spark executors with 8GB+ heaps, G1GC usually reduces maximum pause times significantly.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week6 - GC and Serialization") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

## Part 1: Diagnosing GC Problems

In [ ]:
# Signs of GC problems — what to look for in Spark UI
print("""
GC Diagnostic Signals in Spark UI:
─────────────────────────────────────────────────────────────
Spark UI → Stages → Task Details columns:

Column                  Problem threshold    What it means
──────────────────────  ─────────────────    ─────────────────────────
GC Time                 > 10% of task time   Too much GC
Duration                Very variable        GC pauses causing outliers
Peak Execution Memory   Near executor memory Close to OOM

Spark UI → Executors tab:
Column                  Problem threshold
──────────────────────  ─────────────────
GC Time (total)         > 5-10% of Task Time total

Log patterns to watch for:
  "GC overhead limit exceeded"  → JVM spending >98% time in GC
  "java.lang.OutOfMemoryError"  → heap exhausted
  "Executor lost" (heartbeat)   → GC pause exceeded heartbeat timeout
─────────────────────────────────────────────────────────────
""")

## Part 2: G1GC Configuration

In [ ]:
print("""
G1GC Configuration (add to spark-submit or SparkSession):
─────────────────────────────────────────────────────────────────────
# Executor JVM options
spark.executor.extraJavaOptions=
  -XX:+UseG1GC
  -XX:+PrintGCDetails
  -XX:+PrintGCDateStamps
  -XX:OnOutOfMemoryError='kill -9 %p'
  -XX:InitiatingHeapOccupancyPercent=35
  -XX:G1HeapRegionSize=32m
  -XX:MaxGCPauseMillis=200

# Driver JVM options  
spark.driver.extraJavaOptions=
  -XX:+UseG1GC
  -XX:+PrintGCDetails

Key G1GC flags explained:
  -XX:InitiatingHeapOccupancyPercent=35
    → Start GC when heap is 35% full (default 45%)
    → More frequent but shorter GC cycles — better for Spark

  -XX:G1HeapRegionSize=32m
    → Each G1 region = 32MB (default is auto, can be as small as 1MB)
    → Larger regions = fewer regions = faster region selection
    → Good for large executors (8GB+)

  -XX:MaxGCPauseMillis=200
    → Target max GC pause 200ms (hint, not guarantee)

  -XX:OnOutOfMemoryError='kill -9 %p'
    → Kill executor on OOM instead of hanging
    → Driver gets "executor lost" and can retry tasks
─────────────────────────────────────────────────────────────────────
""")

## Part 3: Serialization — Java vs Kryo

In [ ]:
# Serialization is used when:
# 1. Shuffling data (write to disk, send over network)
# 2. Caching with serialized storage level (MEMORY_ONLY_SER)
# 3. Sending objects from driver to executors (broadcast vars, closures)
# 4. RDD operations across partitions

# Demonstrate the size difference
import pickle
import sys

data = {"name": "Alice", "salary": 95000, "dept": "Engineering", "id": 42}
data_list = [data.copy() for _ in range(1000)]

# Python pickle (what PySpark uses for UDF closures)
pickled = pickle.dumps(data_list)
print(f"Python pickle of 1000 dicts: {len(pickled):,} bytes")

# String representation size
import json
json_bytes = json.dumps(data_list).encode()
print(f"JSON of 1000 dicts:          {len(json_bytes):,} bytes")

print()
print("""
Serialization comparison (JVM side):
─────────────────────────────────────────────────────────────
Format          Size         Speed     When Spark uses it
────────────    ─────        ─────     ─────────────────────
Java            Large        Slow      Default (when no other)
Kryo            ~2-5x smaller  Fast    After enabling (see below)
Tungsten binary Smallest    Fastest   DataFrame operations (automatic)
Arrow           Columnar    Very fast Pandas UDFs, Spark Connect
─────────────────────────────────────────────────────────────
DataFrame ops already use Tungsten binary — no config needed.
Kryo helps for: RDD operations, Python UDF closures, broadcast vars.
""")

In [ ]:
# Kryo configuration
print("""
Enabling Kryo Serialization:
─────────────────────────────────────────────────────────────────
SparkSession.builder
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    .config("spark.kryo.registrationRequired", "false")  # or True
    .config("spark.kryo.classesToRegister",
            "com.mycompany.MyClass,com.mycompany.OtherClass")

spark.kryo.registrationRequired:
  false (default) → unknown classes use Java serialization as fallback
  true  → unregistered classes throw exception
           Forces you to register all classes → max efficiency

When is Kryo used?
  • RDD.map/filter/etc with Python UDF closures (via Pickle on PySpark side)
  • Broadcast variables sent to executors
  • Shuffle data for RDD operations (not DataFrame — Tungsten handles those)
  • Spark Streaming (micro-batch state)

PySpark note: Python UDF data crosses the Python↔JVM boundary via Pickle,
not Kryo. Kryo only affects the JVM side (Java/Scala objects). For PySpark
perf, prefer Pandas UDFs (Arrow) over Python UDFs (Pickle).
─────────────────────────────────────────────────────────────────
""")

## Part 4: Object Creation — Reducing GC Pressure in Code

In [ ]:
import time

# ANTI-PATTERN: Python UDF creates many Python objects per row
# Each Row deserialized from JVM → Python obj, processed, serialized back

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def classify_py(amount):  # creates a Python str per row
    if amount is None: return "Unknown"
    return "High" if amount >= 1000 else "Medium" if amount >= 200 else "Low"

df = spark.range(200_000).withColumn("amount", F.rand() * 2000)

t0 = time.time()
df.withColumn("tier", classify_py(F.col("amount"))).count()
udf_time = time.time() - t0

t0 = time.time()
df.withColumn("tier",
    F.when(F.col("amount") >= 1000, "High")
     .when(F.col("amount") >= 200, "Medium")
     .otherwise("Low")
).count()
builtin_time = time.time() - t0

print(f"Python UDF (creates 200k Python objects): {udf_time:.3f}s")
print(f"Built-in when() (no Python objects):      {builtin_time:.3f}s")
print(f"Speedup: {udf_time/builtin_time:.1f}x")
print()
print("The UDF is slower partly because of Python↔JVM serialization")
print("and partly because of the extra object creation pressure on GC.")

In [ ]:
# Reduce GC pressure: choose serialized storage levels
from pyspark.storagelevel import StorageLevel

large = spark.range(1_000_000).withColumn("v", F.rand())

# MEMORY_ONLY → Java objects on heap → more GC pressure
large.persist(StorageLevel.MEMORY_ONLY)
large.count()
print("MEMORY_ONLY cached (Java objects — more GC pressure)")
large.unpersist()

# MEMORY_ONLY_SER → Kryo bytes → fewer objects → less GC
# Tradeoff: must deserialize on access (CPU cost vs GC benefit)
large.persist(StorageLevel.MEMORY_ONLY)
large.count()
print("MEMORY_ONLY_SER cached (serialized bytes — less GC pressure, slower access)")
large.unpersist()

print("""
Rule: Use MEMORY_ONLY_SER (serialized) when:
  • GC pauses visible in Spark UI are > 10% of task time
  • Dataset is large and access patterns are sequential (not random)
Use MEMORY_ONLY (deserialized) when:
  • Iterative algorithms with frequent random access (ML)
  • DataFrame operations (Tungsten already efficient)
""")

## Part 5: GC Tuning Checklist

In [ ]:
print("""
GC Tuning Checklist (apply in order):
═══════════════════════════════════════════════════════════════
Step 1: Diagnose first
  □ Check Spark UI → Executors → GC Time column
  □ Check Spark UI → Stages → Task Details → GC Time per task
  □ Enable GC logging: -XX:+PrintGCDetails -XX:+PrintGCDateStamps
  □ GC > 10% of task time → problem. GC < 5% → probably fine.

Step 2: Switch to G1GC (most impactful change)
  □ spark.executor.extraJavaOptions = -XX:+UseG1GC
  □ Add -XX:InitiatingHeapOccupancyPercent=35

Step 3: Enable off-heap (for large execution operations)
  □ spark.memory.offHeap.enabled = true
  □ spark.memory.offHeap.size = 2g (or more)

Step 4: Reduce object creation
  □ Replace Python UDFs with built-in functions
  □ Replace Python UDFs with Pandas UDFs
  □ Use MEMORY_ONLY_SER for caching (if random access not needed)

Step 5: Enable Kryo serialization
  □ spark.serializer = org.apache.spark.serializer.KryoSerializer
  □ spark.kryoserializer.buffer.max = 512m

Step 6: Adjust memory regions
  □ Reduce spark.memory.storageFraction if cache not heavily used
     → More memory for execution → fewer GC triggers
  □ Increase executor memory if budget allows
═══════════════════════════════════════════════════════════════
""")

## Exercises

1. A Spark UI shows an executor spending 25% of its time in GC. List the config changes you'd make, in priority order.
2. What is the difference between Minor GC and Major/Full GC? Which one causes task timeouts in Spark?
3. Why doesn't Kryo serialization help with DataFrame operation performance? What does handle it?
4. Write a notebook cell that creates a DataFrame, caches it with `MEMORY_ONLY_SER`, and measures the time difference vs `MEMORY_ONLY` on repeated access.
5. When would you use `-XX:InitiatingHeapOccupancyPercent=35` vs the default 45%? What's the tradeoff?

In [ ]:
# Exercise 4: MEMORY_ONLY vs MEMORY_ONLY_SER timing
test_df = spark.range(500_000).withColumn("v", F.rand())
query = lambda df: df.filter(F.col("v") > 0.5).agg(F.sum("v")).collect()

for level, label in [(StorageLevel.MEMORY_ONLY, "MEMORY_ONLY (deserialized)"),
                     (StorageLevel.MEMORY_ONLY, "MEMORY_ONLY_SER (serialized)")]:
    test_df.persist(level)
    test_df.count()  # materialize
    
    t0 = time.time()
    for _ in range(3):
        query(test_df)
    elapsed = time.time() - t0
    print(f"{label}: 3 queries in {elapsed:.3f}s ({elapsed/3:.3f}s avg)")
    test_df.unpersist()